In [3]:
# ============================================================
# 03_feature_engineering.ipynb
# Feature engineering: super_genre, one-hot, очистка
# ============================================================
# Время работы: ~2 минуты
# Входные данные: data/01_raw_enriched.pkl
# Выходные данные: data/03_features.pkl
# ============================================================

# @title 3.1. Импорты и загрузка
import pandas as pd
import numpy as np
import datetime
import os

os.makedirs('data', exist_ok=True)

df = pd.read_pickle('data/01_raw_enriched.pkl')
print(f"✅ Загружено: {df.shape}")

# @title 3.2. Маппинг жанров в super_genre
def map_genre_to_super_genre(genre):
    genre = str(genre).lower().strip()

    electronic_pop = [
        'pop', 'dance', 'electro', 'edm', 'house', 'techno', 'trance',
        'dubstep', 'drum and bass', 'synth', 'disco', 'k-pop', 'j-pop',
        'chill', 'lo-fi', 'lofi', 'ambient', 'vaporwave', 'idm',
    ]
    rock_alt = [
        'rock', 'punk', 'metal', 'grunge', 'alternative', 'indie',
        'new wave', 'shoegaze', 'emo', 'hardcore',
    ]
    hiphop_rnb = [
        'hip hop', 'hip-hop', 'rap', 'r&b', 'rnb', 'soul', 'trap',
        'funk', 'neo soul', 'g-funk',
    ]
    acoustic_folk = [
        'acoustic', 'folk', 'country', 'blues', 'jazz',
        'singer-songwriter', 'americana', 'bluegrass', 'gospel',
        'classical', 'piano', 'guitar',
    ]

    # ВАЖНО: 'pop' проверяем последним (он перехватывает всё)
    if any(g in genre for g in hiphop_rnb):
        return 'Hip-Hop/R&B/Soul'
    elif any(g in genre for g in rock_alt):
        return 'Rock/Alternative'
    elif any(g in genre for g in acoustic_folk):
        return 'Acoustic/Folk/Country'
    elif any(g in genre for g in electronic_pop):
        return 'Electronic/Pop'
    else:
        return 'World/Classical/Other'

df['super_genre'] = df['genre'].apply(map_genre_to_super_genre)
print("\nsuper_genre:")
print(df['super_genre'].value_counts())

# @title 3.3. Album age
current_year = datetime.datetime.now().year
df['album_age_years'] = current_year - df['album_year']

print(f"\nalbum_age_years:")
print(df['album_age_years'].describe())

# @title 3.4. Замена Deezer-нулей на NaN
for col in ['deezer_bpm', 'deezer_gain', 'deezer_rank',
            'deezer_artist_fans', 'deezer_artist_albums']:
    if col in df.columns:
        n_zero = (df[col] == 0).sum()
        df[col] = df[col].replace(0, np.nan)
        print(f"  {col}: заменено {n_zero} нулей на NaN")

# @title 3.5. Исправление аномалий MusicBrainz (Piano 1959)
mask_suspicious = (
    (df['album'] == 'Piano') &
    (df['album_year'] == 1959) &
    (df['deezer_bpm'] > 0)
)
print(f"\nПодозрительных 'Piano' 1959: {mask_suspicious.sum()}")
df.loc[mask_suspicious, 'album_year'] = np.nan
df.loc[mask_suspicious, 'album_age_years'] = np.nan
df = df[~mask_suspicious].reset_index(drop=True)
print(f"Удалено {mask_suspicious.sum()} строк, осталось {len(df)}")

# @title 3.6. Группировка стран (топ-10 + Other)
for col in ['release_country', 'artist_country']:
    n_before = df[col].nunique()
    unknown_pct = (df[col] == 'unknown').mean()

    top_by_count = df[col].value_counts().head(10).index.tolist()
    valid = [c for c in top_by_count if df[df[col] == c].shape[0] >= 10]

    # Сохранить unknown, если информативен
    if 'unknown' in df[col].values:
        unknown_pop = df[df[col] == 'unknown']['popularity'].mean()
        overall_pop = df['popularity'].mean()
        if abs(unknown_pop - overall_pop) > 3:
            valid.append('unknown')
            print(f"  {col}: 'unknown' информативен "
                  f"(pop {unknown_pop:.1f} vs {overall_pop:.1f})")
        elif unknown_pct > 0.1:
            valid.append('unknown')

    df[col] = df[col].where(df[col].isin(valid), 'Other')
    print(f"{col}: {n_before} → {df[col].nunique()} категорий")

# @title 3.7. Удаление треков без Deezer-данных
n_before = len(df)
df = df[df['deezer_rank'].notna()].reset_index(drop=True)
print(f"\nУдалено {n_before - len(df)} строк без Deezer, осталось {len(df)}")

# @title 3.8. Удаление идентификаторов и дубликатов
drop_cols = [
    # Идентификаторы
    'id', 'name', 'artists', 'album', 'genre',
    # Служебные
    'deezer_found', 'release_label',
    # Дубликаты
    'album_year',           # = current_year - album_age_years
    'deezer_duration',      # = duration_ms / 1000
    'deezer_explicit',      # = explicit
    'has_musicbrainz_data',         # константа
    'has_musicbrainz_artist_data'  # константа
]

df_encoded = df.drop(columns=drop_cols, errors='ignore')

# Удалить deezer_rank (data leakage)
df_encoded = df_encoded.drop(columns=['deezer_rank'], errors='ignore')
print(f"После удаления deezer_rank: {df_encoded.shape}")

# Привести числовые столбцы к числу
numeric_cols = [
    'deezer_bpm', 'deezer_gain', 'deezer_rank',
    'deezer_artist_fans', 'deezer_artist_albums',
]
for col in numeric_cols:
    if col in df_encoded.columns:
        df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce')

print(f"\nПосле удаления: {df_encoded.shape}")

# @title 3.9. Булевы → int
for col in ['explicit', 'artist_is_active', 'is_unofficial_release']:
    if col in df_encoded.columns and df_encoded[col].dtype == 'bool':
        df_encoded[col] = df_encoded[col].astype(int)

# @title 3.10. Флаги для структурных пропусков
df_encoded['career_duration_known'] = df_encoded[
    'artist_career_duration_years'
].notna().astype(int)
df_encoded['album_age_known'] = df_encoded[
    'album_age_years'
].notna().astype(int)

# @title 3.11. One-hot
cat_cols = [
    'release_country', 'release_status', 'artist_type',
    'artist_country', 'super_genre'
]
cat_cols = [c for c in cat_cols if c in df_encoded.columns]

n_before = df_encoded.shape[1]
df_encoded = pd.get_dummies(
    df_encoded, columns=cat_cols, drop_first=True, dtype=int
)
print(f"One-hot: {n_before} → {df_encoded.shape[1]} столбцов")

# @title 3.12. Проверка
obj_cols = df_encoded.select_dtypes(include='object').columns.tolist()
print(f"Object-столбцы: {obj_cols if obj_cols else 'нет ✅'}")

# Константы
constants = [c for c in df_encoded.columns if df_encoded[c].nunique() <= 1]
print(f"Константные столбцы: {constants if constants else 'нет ✅'}")

print(f"\nФинальный df_encoded: {df_encoded.shape}")
print(f"Пропусков: {df_encoded.isna().sum().sum()}")

# @title 3.13. Сохранение
df_encoded.to_pickle('data/03_features.pkl')
df_encoded.to_csv('data/03_features.csv', index=False)

with open('data/03_features_columns.txt', 'w') as f:
    for col in df_encoded.columns:
        f.write(f"{col}\n")

print(f"\n{'=' * 60}")
print(f"✅ 03_feature_engineering ЗАВЕРШЁН")
print(f"{'=' * 60}")
print(f"Размер: {df_encoded.shape}")

✅ Загружено: (6300, 30)

super_genre:
super_genre
World/Classical/Other    3150
Electronic/Pop           1400
Rock/Alternative         1050
Acoustic/Folk/Country     550
Hip-Hop/R&B/Soul          150
Name: count, dtype: int64

album_age_years:
count    5375.000000
mean       13.987721
std        12.109806
min         0.000000
25%         5.000000
50%        10.000000
75%        20.000000
max        91.000000
Name: album_age_years, dtype: float64
  deezer_bpm: заменено 3717 нулей на NaN
  deezer_gain: заменено 69 нулей на NaN
  deezer_rank: заменено 3 нулей на NaN
  deezer_artist_fans: заменено 40 нулей на NaN
  deezer_artist_albums: заменено 33 нулей на NaN

Подозрительных 'Piano' 1959: 3
Удалено 3 строк, осталось 6297
release_country: 71 → 11 категорий
  artist_country: 'unknown' информативен (pop 25.1 vs 30.8)
artist_country: 74 → 11 категорий

Удалено 1062 строк без Deezer, осталось 5235
После удаления deezer_rank: (5235, 19)

После удаления: (5235, 19)
One-hot: 21 → 51 столбцов
Obj

/tmp/ipykernel_2969/4020094063.py:72: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace(0, np.nan)



✅ 03_feature_engineering ЗАВЕРШЁН
Размер: (5235, 51)
